# Fine Print on a free T4 GPU (Google Colab)

Runs the whole app on Colab's NVIDIA T4 with local models through Ollama: no API key, nothing leaves the notebook's machine.

1. **Runtime > Change runtime type > T4 GPU**, then run the cells in order.
2. Setup takes about 5 minutes the first time (installing Ollama and downloading the models).
3. The last cell prints a public link to the app.

In [ ]:
# 1. Check the GPU (should show Tesla T4, 15-16 GB)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the project
Upload `ma_fineprint_engine.zip` when prompted. (Or put it in Google Drive and change the path in the second option.)

In [ ]:
import os, zipfile
from google.colab import files

if not os.path.isdir("/content/ma_fineprint_engine"):
    uploaded = files.upload()                       # choose ma_fineprint_engine.zip
    name = next(iter(uploaded))
    zipfile.ZipFile(name).extractall("/content")
    # Option B - from Google Drive instead of uploading:
    # from google.colab import drive; drive.mount('/content/drive')
    # zipfile.ZipFile('/content/drive/MyDrive/ma_fineprint_engine.zip').extractall('/content')
%cd /content/ma_fineprint_engine
!grep -E '^(LLM_PROVIDER|OLLAMA_MODEL|OLLAMA_CONTEXT)' utils.py

## 3. Install Ollama, pull the models, install Python packages
Uses the settings in `utils.py` (default: `qwen3:14b` + `nomic-embed-text`, 16k context, flash attention, 8-bit KV cache).

In [ ]:
!bash scripts/setup_t4.sh

## 4. Check speed
Both lines under the speed figures should say **100% on GPU**. If not, lower `OLLAMA_CONTEXT` or switch to `qwen3:8b` in `utils.py` and re-run step 3.

In [ ]:
!python3 scripts/benchmark_ollama.py

## 5. Start the app with a public link
Open the link that appears. It stays up while this notebook is connected.

In [ ]:
import subprocess, time, re, pathlib
subprocess.Popen(["bash", "scripts/start_app.sh", "--public"], stdout=open("start.log", "w"), stderr=subprocess.STDOUT,
                 start_new_session=True)
for _ in range(90):
    log = pathlib.Path("tunnel.log").read_text() if pathlib.Path("tunnel.log").exists() else ""
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
    if m:
        print("Open the app:", m.group(0)); break
    time.sleep(2)
else:
    print("No link yet - see start.log and tunnel.log")

### Troubleshooting
- `!tail -n 30 ollama.log` shows the model server log; `!ollama ps` shows what is loaded and how much is on the GPU.
- Out-of-memory errors: set `OLLAMA_CONTEXT = 12288` or `OLLAMA_MODEL = "qwen3:8b"` in `utils.py`, then re-run steps 3 and 5.